**Reasoning**:
Importing the necessary libraries, defining the dual-view data augmentation pipeline, and creating the CIFAR-10 DataLoader with these dual views.



In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms
import torchvision.datasets as datasets

class DualViewTransform:
    def __init__(self, transform):
        self.transform = transform

    def __call__(self, x):
        view1 = self.transform(x)
        view2 = self.transform(x)
        return view1, view2

# Define standard augmentations for BYOL on CIFAR-10
byol_transform = transforms.Compose([
    transforms.RandomResizedCrop(32, scale=(0.2, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomApply([transforms.ColorJitter(0.4, 0.4, 0.4, 0.1)], p=0.8),
    transforms.RandomGrayscale(p=0.2),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

dual_view_transform = DualViewTransform(byol_transform)

# Download and load CIFAR-10 training set
train_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=dual_view_transform)
train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True, num_workers=2, drop_last=True)

print(f"Successfully loaded dataset of size: {len(train_dataset)}")
print(f"Number of batches in DataLoader: {len(train_loader)}")

100%|██████████| 170M/170M [00:13<00:00, 12.2MB/s]


Successfully loaded dataset of size: 50000
Number of batches in DataLoader: 195


# Task
Implement the self-supervised Bootstrap Your Own Latent (BYOL) model architecture in PyTorch, including the Online Network (with ResNet backbone, projection head, and predictor), the Target Network, the Exponential Moving Average (EMA) update helper, the normalized mean squared error loss function, and a training loop verified on the dual-view CIFAR-10 dataset.

## Define BYOL Model Architecture

### Subtask:
Implement the Online and Target networks with a ResNet backbone modified for CIFAR-10, projection heads, and the online predictor network.


**Reasoning**:
We will implement the EMA parameter update class, the symmetric normalized mean squared error loss function for BYOL, and verify them using dummy tensors.



In [3]:
class EMAUpdater:
    def __init__(self, tau=0.99):
        self.tau = tau

    def update(self, online_net, target_net):
        for param_online, param_target in zip(online_net.parameters(), target_net.parameters()):
            param_target.data = self.tau * param_target.data + (1 - self.tau) * param_online.data

def byol_loss_fn(p, z):
    p = F.normalize(p, dim=-1, p=2)
    z = F.normalize(z, dim=-1, p=2)
    return 2 - 2 * (p * z).sum(dim=-1)

def symmetric_byol_loss(p1, z2, p2, z1):
    loss_1 = byol_loss_fn(p1, z2)
    loss_2 = byol_loss_fn(p2, z1)
    return (loss_1 + loss_2).mean()

# Verification with dummy tensors
mock_p1 = torch.randn(4, 256)
mock_z2 = torch.randn(4, 256)
mock_p2 = torch.randn(4, 256)
mock_z1 = torch.randn(4, 256)

loss_value = symmetric_byol_loss(mock_p1, mock_z2, mock_p2, mock_z1)
print(f"Verified BYOL loss function on dummy tensors. Loss output: {loss_value.item():.4f}")

Verified BYOL loss function on dummy tensors. Loss output: 4.0322


**Reasoning**:
We need to implement the online and target networks using a modified ResNet-18 backbone suitable for 32x32 CIFAR-10 images, along with the projection heads and the online predictor.



In [2]:
import torchvision.models as models

class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(MLP, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, output_dim)
        )

    def forward(self, x):
        return self.net(x)

class BYOLBackbone(nn.Module):
    def __init__(self, base_model="resnet18", out_dim=256, projection_hidden_dim=4096):
        super(BYOLBackbone, self).__init__()

        # Load ResNet18 and modify it for 32x32 CIFAR-10 images
        if base_model == "resnet18":
            resnet = models.resnet18(weights=None)
            resnet.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
            resnet.maxpool = nn.Identity() # Remove max pooling to preserve resolution

            self.feature_dim = resnet.fc.in_features
            # Extract layers up to average pooling
            self.encoder = nn.Sequential(*list(resnet.children())[:-1])
        else:
            raise ValueError(f"Unsupported backbone: {base_model}")

        # Projection head
        self.projection_head = MLP(self.feature_dim, projection_hidden_dim, out_dim)

    def forward(self, x):
        x = self.encoder(x)
        x = torch.flatten(x, start_dim=1)
        x = self.projection_head(x)
        return x

# Helper wrapper for Online network (which includes the predictor)
class OnlineNetwork(nn.Module):
    def __init__(self, backbone, out_dim=256, predictor_hidden_dim=4096):
        super(OnlineNetwork, self).__init__()
        self.backbone = backbone
        self.predictor = MLP(out_dim, predictor_hidden_dim, out_dim)

    def forward(self, x):
        representation = self.backbone(x)
        prediction = self.predictor(representation)
        return prediction

# Initialize Online and Target Networks
online_backbone = BYOLBackbone()
online_net = OnlineNetwork(online_backbone)
target_net = BYOLBackbone() # Target network does not have a predictor

# Copy online parameters to target parameters initially
for param_online, param_target in zip(online_net.backbone.parameters(), target_net.parameters()):
    param_target.data.copy_(param_online.data)
    param_target.requires_grad = False # Target network parameters are not updated by gradient descent

print("Online and Target networks initialized successfully.")
print(f"Online Network Architecture:\n{online_net}")

Online and Target networks initialized successfully.
Online Network Architecture:
OnlineNetwork(
  (backbone): BYOLBackbone(
    (encoder): Sequential(
      (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): Identity()
      (4): Sequential(
        (0): BasicBlock(
          (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
          (relu): ReLU(inplace=True)
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        )
        (1): BasicBlock(
          (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding

## Implement EMA and Loss Function

### Subtask:
Implement the Exponential Moving Average (EMA) parameter update helper and the normalized mean squared error loss function for BYOL.


## Create Training Loop

### Subtask:
Set up a PyTorch training loop for self-supervised learning with Online and Target parameter updates on CIFAR-10.


**Reasoning**:
Setting up and running the PyTorch training loop on the active device for 1 epoch of pre-training to verify the BYOL implementation.



In [13]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Move networks to the device
online_net = online_net.to(device)
target_net = target_net.to(device)

# Optimizer (using Adam for quick convergence verification during testing)
optimizer = optim.Adam(online_net.parameters(), lr=3e-4)

# Initialize EMA Updater
ema_updater = EMAUpdater(tau=0.99)

# Set models to training mode
online_net.train()
target_net.train()  # Although target parameters are updated via EMA, BatchNorm stats can update

print("Starting 1 epoch of pre-training...")
epoch_loss = 0.0
for batch_idx, ((view1, view2), _) in enumerate(train_loader):
    # Move dual-view images to device
    view1 = view1.to(device)
    view2 = view2.to(device)

    # Zero the gradients
    optimizer.zero_grad()

    # Online network predictions
    p1 = online_net(view1)
    p2 = online_net(view2)

    # Target network projections (with no gradients)
    with torch.no_grad():
        z1 = target_net(view1).detach()
        z2 = target_net(view2).detach()

    # Calculate symmetric BYOL loss
    loss = symmetric_byol_loss(p1, z2, p2, z1)

    # Backpropagation
    loss.backward()
    optimizer.step()

    # Target network parameter updates using EMA
    ema_updater.update(online_net.backbone, target_net)

    epoch_loss += loss.item()

    if (batch_idx + 1) % 10 == 0:
        print(f"Batch [{batch_idx+1}/{len(train_loader)}] - Loss: {loss.item():.4f}")

average_loss = epoch_loss / len(train_loader)
print(f"Completed 1 epoch! Average Pre-training Loss: {average_loss:.4f}")

Using device: cuda
Starting 1 epoch of pre-training...
Batch [10/195] - Loss: 0.5686
Batch [20/195] - Loss: 0.5261
Batch [30/195] - Loss: 0.5469
Batch [40/195] - Loss: 0.5783
Batch [50/195] - Loss: 0.5021
Batch [60/195] - Loss: 0.5067
Batch [70/195] - Loss: 0.5266
Batch [80/195] - Loss: 0.5064
Batch [90/195] - Loss: 0.5380
Batch [100/195] - Loss: 0.5327
Batch [110/195] - Loss: 0.5556
Batch [120/195] - Loss: 0.5811
Batch [130/195] - Loss: 0.4850
Batch [140/195] - Loss: 0.5255
Batch [150/195] - Loss: 0.5799
Batch [160/195] - Loss: 0.4815
Batch [170/195] - Loss: 0.4793
Batch [180/195] - Loss: 0.5134


KeyboardInterrupt: 